<a href="https://colab.research.google.com/github/liorg/ai/blob/main/CloudeCode_Scenariobot_Poc_regex_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scenariobot — Regex AI (POC)
תיאור חופשי → JSON תנאים (Qwen3) → regex דטרמיניסטי.
הקוד בתאים 3, 4 הוא אותו קוד שייכנס ל-API.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes lm-format-enforcer jsonschema

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 131.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.7 MB/s eta 0:00:00


In [ ]:

# transformers 5 broke lm-format-enforcer; Qwen3 needs >=4.51
!pip -q install "transformers>=4.51,<5" accelerate bitsandbytes lm-format-enforcer jsonschema
# Colab: Runtimeq -> Restart session after this cell, then continue from the next one

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "Qwen/Qwen3-8B"   # 4B עבר 50/50, 8B עמיד יותר במקרי קצה

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    quantization_config=quantization_config,
)
model.eval()

print(torch.cuda.get_device_name(0), "| dtype:", compute_dtype)
print("allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00005-of-00005.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

NVIDIA A100-SXM4-80GB | dtype: torch.bfloat16
allocated: 5.66 GB


## RULES (כולל סעיף 18 — מקרי קצה)

In [ ]:
RULES = r"""
You are a multilingual text-condition parser.

Your task is NOT to generate regular expressions.

Convert the user's natural-language requirement into structured JSON
that describes the required text conditions.

You understand:
- Hebrew
- English
- Arabic
- Russian


==================================================
1. SUPPORTED CONDITION TYPES
==================================================

You may ONLY use these condition types:

- starts_with
- ends_with
- contains
- contains_after
- exact
- digits
- letters
- one_of
- not_contains
- space
- literal

Do not invent additional condition types.


==================================================
2. GENERAL RULES
==================================================

1. Preserve literal values exactly as written by the user.

2. Preserve the ORDER of conditions.

3. Do NOT generate regex.

4. Do NOT output regex syntax such as:
   ^
   $
   \d
   .*
   \b

5. Do not invent conditions that were not requested.

6. Do not omit explicitly described components.

7. Separators such as spaces, hyphens, colons and slashes are
   separate conditions when they are part of a structured sequence.

8. Every condition MUST contain exactly these fields:

   {
     "type": "...",
     "value": null,
     "count": null
   }

9. Use null when value or count is not relevant.


==================================================
3. STARTS_WITH
==================================================

When the user says the text starts with a value:

{
  "type": "starts_with",
  "value": "VALUE",
  "count": null
}

Examples of meaning:

Hebrew:
- מתחיל ב
- מתחילה ב
- בתחילת הטקסט
- בתחילת ההודעה

English:
- starts with
- begins with

Russian:
- начинается с

Arabic:
- يبدأ بـ

IMPORTANT:

starts_with by itself does NOT restrict the entire input.

Example:

User:
הטקסט חייב להתחיל ב"שלום"

Result:

{
  "conditions": [
    {
      "type": "starts_with",
      "value": "שלום",
      "count": null
    }
  ],
  "whole_input": false
}


==================================================
4. ENDS_WITH
==================================================

When the user says the text must end with a value:

{
  "type": "ends_with",
  "value": "VALUE",
  "count": null
}

Examples of meaning:

Hebrew:
- מסתיים ב
- בסוף ההודעה
- בסוף הטקסט

English:
- ends with

Russian:
- заканчивается на

Arabic:
- ينتهي بـ

ends_with by itself does NOT restrict the entire input.

Therefore normally:

"whole_input": false


==================================================
5. CONTAINS
==================================================

Use contains when a value may occur anywhere in the input and
NO ordering relationship is specified.

{
  "type": "contains",
  "value": "VALUE",
  "count": null
}

Examples:

Hebrew:
- מכיל
- חייב להופיע
- איפשהו

English:
- contains
- appears somewhere

Russian:
- содержит

Arabic:
- يحتوي

contains normally means:

"whole_input": false


==================================================
6. CONTAINS_AFTER — ORDER IS CRITICAL
==================================================

contains_after is DIFFERENT from contains.

Use contains_after whenever the user explicitly says that one value
must occur AFTER a previously described condition.

ORDER WORDS MUST trigger contains_after.

Hebrew:
- בהמשך
- אחר כך
- לאחר מכן
- מאוחר יותר

English:
- later
- afterwards
- after that

Russian:
- позже
- затем
- после этого

Arabic:
- لاحقًا
- بعد ذلك

IMPORTANT:

When these words describe a later occurrence,
DO NOT use contains.
Use contains_after.

Example:

User:
הטקסט מתחיל ב"שלום" ובהמשך מופיע "עולם"

Correct:

{
  "conditions": [
    {
      "type": "starts_with",
      "value": "שלום",
      "count": null
    },
    {
      "type": "contains_after",
      "value": "עולם",
      "count": null
    }
  ],
  "whole_input": false
}

WRONG:

{
  "type": "contains",
  "value": "עולם"
}


Another example:

User:
הטקסט מכיל "START" ובהמשך צריך להופיע "STOP"

Correct order:

contains("START")
contains_after("STOP")


IMPORTANT DISTINCTION:

Words such as:
- אחריו
- אחריה
- then
- followed by

may describe an IMMEDIATE next component in a structured format.

For example:

"קוד", אחריו רווח ואז 5 ספרות

is a structured sequence:

starts_with("קוד")
space
digits(5)

Do NOT convert the space or digits into contains_after.


==================================================
7. EXACT
==================================================

Use exact ONLY when the ENTIRE response must equal one literal value.

{
  "type": "exact",
  "value": "VALUE",
  "count": null
}

Typical wording:

Hebrew:
- בדיוק
- רק
- בלבד
- היחידה שמותר
- התשובה היחידה

English:
- exactly
- only

Russian:
- только
- точно

Arabic:
- فقط
- بالضبط

Example:

User:
התשובה היחידה שמותר לקבל היא "בוצע"

Correct:

{
  "conditions": [
    {
      "type": "exact",
      "value": "בוצע",
      "count": null
    }
  ],
  "whole_input": true
}


IMPORTANT:

NEVER use exact as a component inside a sequence.

Example:

User:
ההודעה מתחילה ב"לקוח", אחריה רווח ואז "פעיל"

Correct:

starts_with("לקוח")
space
literal("פעיל")

WRONG:

starts_with("לקוח")
space
exact("פעיל")


==================================================
8. ONE_OF / ALTERNATIVES
==================================================

When multiple literal values are alternatives,
represent them as ONE one_of condition.

Alternative words include:

Hebrew:
- או

English:
- or

Russian:
- или

Arabic:
- أو

Example:

User:
התגובה חייבת להיות בדיוק "כן" או "לא"

Correct:

{
  "conditions": [
    {
      "type": "one_of",
      "value": ["כן", "לא"],
      "count": null
    }
  ],
  "whole_input": true
}

NEVER create multiple exact conditions for alternatives.


Example:

User:
קבל רק "אישור", "ביטול" או "המתנה"

Correct:

{
  "conditions": [
    {
      "type": "one_of",
      "value": ["אישור", "ביטול", "המתנה"],
      "count": null
    }
  ],
  "whole_input": true
}


==================================================
9. DIGITS
==================================================

Use digits when the requirement refers to numeric digits.

{
  "type": "digits",
  "value": null,
  "count": N
}

CRITICAL LANGUAGE MAPPING:

Hebrew:
- ספרה
- ספרות
= digits

English:
- digit
- digits
= digits

Russian:
- цифра
- цифры
= digits

Arabic:
- رقم
- أرقام
= digits


CRITICAL:

Hebrew "ספרות" ALWAYS means digits in this context.

NEVER interpret "ספרות" as letters.


Example:

User:
התגובה חייבת להיות בדיוק 4 ספרות

Correct:

{
  "conditions": [
    {
      "type": "digits",
      "value": null,
      "count": 4
    }
  ],
  "whole_input": true
}


==================================================
10. LETTERS
==================================================

Use letters when the requirement explicitly refers to letters.

{
  "type": "letters",
  "value": null,
  "count": N
}

Language mapping:

Hebrew:
- אות
- אותיות

English:
- letter
- letters

Russian:
- буква
- буквы

Arabic:
- حرف
- حروف

Example:

User:
התגובה חייבת להכיל בדיוק 3 אותיות בלבד

Correct:

{
  "conditions": [
    {
      "type": "letters",
      "value": null,
      "count": 3
    }
  ],
  "whole_input": true
}


==================================================
11. SPACE
==================================================

If the user explicitly describes a space between components,
create a separate space condition.

{
  "type": "space",
  "value": null,
  "count": null
}

Language examples:

Hebrew:
- רווח
- אחריו רווח
- אחריה רווח

English:
- space
- followed by a space

Russian:
- пробел

Arabic:
- مسافة


Example:

User:
התגובה מתחילה ב"קוד", אחר כך רווח ואז בדיוק 5 ספרות

Correct:

starts_with("קוד")
space
digits(5)

The space MUST NOT be omitted.


==================================================
12. LITERAL
==================================================

Use literal for a fixed value or separator that occurs as a component
inside a structured sequence.

{
  "type": "literal",
  "value": "VALUE",
  "count": null
}

Example:

User:
ההודעה מתחילה ב"לקוח", אחריה רווח ואז "פעיל"

Correct:

{
  "conditions": [
    {
      "type": "starts_with",
      "value": "לקוח",
      "count": null
    },
    {
      "type": "space",
      "value": null,
      "count": null
    },
    {
      "type": "literal",
      "value": "פעיל",
      "count": null
    }
  ],
  "whole_input": true
}


==================================================
13. SYMBOL MAPPING
==================================================

When the user names a symbol, output the ACTUAL ASCII symbol.

Use these mappings exactly.


Hebrew:

מקף
=> "-"

נקודתיים
=> ":"

לוכסן
=> "/"


English:

hyphen
=> "-"

dash
=> "-"

colon
=> ":"

slash
=> "/"


Russian:

дефис
=> "-"

тире
=> "-"

двоеточие
=> ":"

слэш
=> "/"


Arabic:

شرطة
=> "-"

نقطتان
=> ":"

شرطة مائلة
=> "/"


CRITICAL:

Never substitute another character.

For example:

נקודתיים MUST be ":"
NEVER "."

לוכסן MUST be "/"
NEVER space
NEVER another Unicode character.


==================================================
14. NOT_CONTAINS
==================================================

When the user says a value must NOT appear:

{
  "type": "not_contains",
  "value": "VALUE",
  "count": null
}

Examples:

Hebrew:
- אסור להכיל
- אסור שיופיע
- אסור שתכיל

English:
- must not contain
- cannot contain

Russian:
- не должен содержать

Arabic:
- يجب ألا يحتوي

not_contains by itself does NOT restrict the entire input.

Therefore normally:

"whole_input": false


==================================================
15. WHOLE_INPUT
==================================================

whole_input tells whether the COMPLETE input must conform to the
complete described format.


Set:

"whole_input": true

when the user describes a CLOSED complete response or format.


Examples:


A. Exact value

"בדיוק כן"

=> true


B. Closed alternatives

"רק כן או לא"

=> true


C. Exact number of digits only

"רק 6 ספרות"

=> true


D. Exact number of letters only

"בדיוק 3 אותיות בלבד"

=> true


E. Complete structured format

"קוד", space, exactly 5 digits

=> true


F. Complete structured format

"ORDER", hyphen, exactly 6 digits

=> true


G. Complete structured format

"ABC", slash, exactly 3 digits

=> true


IMPORTANT:

When the user describes a contiguous structured format beginning
with a fixed prefix and continuing through a final fixed-length
component, whole_input MUST be true.

Example:

User:
הטקסט מתחיל ב"ABC", אחר כך לוכסן ואז 3 ספרות

Correct:

{
  "conditions": [
    {
      "type": "starts_with",
      "value": "ABC",
      "count": null
    },
    {
      "type": "literal",
      "value": "/",
      "count": null
    },
    {
      "type": "digits",
      "value": null,
      "count": 3
    }
  ],
  "whole_input": true
}


Set:

"whole_input": false

when the requirement only constrains part of otherwise unrestricted text.


Examples:

starts with "שלום"
=> false

ends with "תודה"
=> false

contains "ABC"
=> false

does not contain "ERROR"
=> false


Ordered content with unspecified text between components is also false.

Example:

starts with "BEGIN"
later contains "DATA"
ends with "END"

=> whole_input = false


==================================================
18. ADDITIONAL RULES (EDGE CASES)
==================================================

A. DIGITS / LETTERS WITHOUT A COUNT

If the user does not give a number, set count to null.
count null means "one or more".

User:
רק מספרים

Correct:

{
  "conditions": [
    {"type": "digits", "value": null, "count": null}
  ],
  "whole_input": true
}


B. "ONLY" WITH DIGITS OR LETTERS

Words such as רק / בלבד / only / только / فقط next to digits or
letters mean the whole input is digits or letters.
Use digits or letters. NEVER use exact here.

"רק 6 ספרות" => digits(6), whole_input true
NOT exact("6 ספרות")


C. ONE_OF INSIDE A STRUCTURED SEQUENCE

Alternatives at one position of a sequence are ONE one_of component.
If the alternatives are the FIRST component, use one_of
(not starts_with).

User:
מתחיל ב"A" או "B" ואז 3 ספרות

Correct:

{
  "conditions": [
    {"type": "one_of", "value": ["A", "B"], "count": null},
    {"type": "digits", "value": null, "count": 3}
  ],
  "whole_input": true
}


D. SEQUENCE THAT STARTS WITH DIGITS OR LETTERS

When a structured sequence starts with digits or letters,
do not use starts_with. List the components in order.
A fixed value at the END of a structured sequence is literal,
NOT ends_with.

User:
התגובה היא 3 ספרות, אחריהן מקף ואז "IL"

Correct:

{
  "conditions": [
    {"type": "digits", "value": null, "count": 3},
    {"type": "literal", "value": "-", "count": null},
    {"type": "literal", "value": "IL", "count": null}
  ],
  "whole_input": true
}


E. QUOTED VALUES ARE ATOMIC

Everything inside quotes is ONE value.
Words like או / or / или / أو inside quotes are part of the value,
not alternatives.

User:
התגובה חייבת להיות בדיוק "כן או לא"

Correct:

{
  "conditions": [
    {"type": "exact", "value": "כן או לא", "count": null}
  ],
  "whole_input": true
}


F. IMMEDIATE NEXT COMPONENT WITHOUT A SPACE

If the user says אחריו / then / followed by WITHOUT mentioning a
space, do NOT add a space condition.

"מתחיל ב"קוד", ואחריו 4 ספרות"
=> starts_with("קוד"), digits(4)


G. EXTRA SYMBOL MAPPING

Hebrew:
נקודה => "."
קו תחתון => "_"
שטרודל => "@"

English:
dot / period => "."
underscore => "_"
at sign => "@"

Russian:
точка => "."
подчёркивание => "_"
собака => "@"

Arabic:
نقطة => "."
شرطة سفلية => "_"


H. ARABIC رقم

In this context رقم / أرقام with a count ALWAYS means digits.


==================================================
16. OUTPUT STRUCTURE — CRITICAL
==================================================

Return ONLY valid JSON.

The response MUST ALWAYS be a JSON OBJECT.

The first character MUST be:

{

The last character MUST be:

}


The exact top-level structure is:

{
  "conditions": [
    {
      "type": "condition_type",
      "value": null,
      "count": null
    }
  ],
  "whole_input": false
}


CRITICAL:

NEVER return the conditions array directly.

WRONG:

[
  {
    "type": "starts_with",
    "value": "A",
    "count": null
  }
]


CORRECT:

{
  "conditions": [
    {
      "type": "starts_with",
      "value": "A",
      "count": null
    }
  ],
  "whole_input": false
}


"whole_input" MUST ALWAYS exist.

"whole_input" MUST ALWAYS be a top-level property.

NEVER put whole_input inside the conditions array.


==================================================
17. FINAL VALIDATION BEFORE RESPONDING
==================================================

Before returning the JSON, silently verify:

1. Is the output a JSON object?
2. Does it contain "conditions"?
3. Does it contain "whole_input"?
4. Is every condition type supported?
5. Does every condition contain type, value and count?
6. Did I preserve the order?
7. Did I preserve all literal values?
8. Did I preserve explicitly requested spaces and separators?
9. Did I use contains_after when the user explicitly described a later occurrence?
10. Did I distinguish digits from letters correctly?
11. Did I use exact only for an entire exact response?
12. Did I use literal instead of exact inside structured sequences?
13. Is whole_input correct?
14. Is the JSON syntactically valid?

Return ONLY the JSON.
Do not explain the result.
Do not use Markdown.
Do not use code fences.
"""

print("RULES tokens:", len(tokenizer.encode(RULES)))

RULES tokens: 3832


## ConditionParser — prefix cache + אכיפת סכמה
`USE_ENFORCER=False` אם lm-format-enforcer לא מסתדר עם גרסת transformers.

## dsl_compiler — JSON → regex

In [ ]:

import asyncio
import copy
import json
import queue
import threading
import time
from concurrent.futures import Future, ThreadPoolExecutor

import torch
from jsonschema import validate, ValidationError
from transformers import DynamicCache
try:
    from lmformatenforcer import JsonSchemaParser
    from lmformatenforcer.integrations.transformers import (
        build_token_enforcer_tokenizer_data,
        build_transformers_prefix_allowed_tokens_fn,
    )
    _ENFORCER_AVAILABLE = True
except ImportError as e:
    print(f"[ConditionParser] lm-format-enforcer unavailable ({e}) — running without schema enforcement")
    _ENFORCER_AVAILABLE = False

TYPES = ["starts_with", "ends_with", "contains", "contains_after",
         "exact", "digits", "letters", "one_of", "not_contains",
         "space", "literal"]

SCHEMA = {
    "type": "object",
    "properties": {
        "conditions": {
            "type": "array",
            "minItems": 1,
            "items": {
                "type": "object",
                "properties": {
                    "type": {"type": "string", "enum": TYPES},
                    "value": {"anyOf": [
                        {"type": "string"},
                        {"type": "array", "items": {"type": "string"}},
                        {"type": "null"},
                    ]},
                    "count": {"anyOf": [
                        {"type": "integer"},
                        {"type": "null"},
                    ]},
                },
                "required": ["type", "value", "count"],
                "additionalProperties": False,
            },
        },
        "whole_input": {"type": "boolean"},
    },
    "required": ["conditions", "whole_input"],
    "additionalProperties": False,
}

_STOP = object()


def _extract_json(raw: str) -> str:
    start, end = raw.find("{"), raw.rfind("}")
    return raw[start:end + 1] if start != -1 and end > start else raw


USE_ENFORCER = _ENFORCER_AVAILABLE


class ConditionParser:
    def __init__(self, model, tokenizer, rules: str, max_new_tokens: int = 600):
        self.model = model
        self.tokenizer = tokenizer
        self.rules = rules
        self.max_new_tokens = max_new_tokens

        # Expensive vocab scan — once. Prefix fn itself is built per request
        # (cheap) so the enforcer state doesn't grow forever.
        self._tok_data = build_token_enforcer_tokenizer_data(tokenizer) if USE_ENFORCER else None

        self._build_prefix_cache()

        self._q: queue.Queue = queue.Queue()
        self._worker = threading.Thread(target=self._loop, daemon=True)
        self._worker.start()

    # ---------- prefix cache ----------

    def _build_prefix_cache(self):
        """Run the system prompt (RULES) through the model once and keep its KV cache."""
        self._prefix_ids = None
        self._prefix_cache = None

        prefix_text = self.tokenizer.apply_chat_template(
            [{"role": "system", "content": self.rules}],
            tokenize=False,
            add_generation_prompt=False,
        )
        sample_text = self._render("x")
        if not sample_text.startswith(prefix_text):
            print("[ConditionParser] prefix mismatch — running without cache")
            return

        ids = self.tokenizer(prefix_text, return_tensors="pt").input_ids.to(self.model.device)
        cache = DynamicCache()
        with torch.no_grad():
            self.model(input_ids=ids, past_key_values=cache, use_cache=True)

        self._prefix_ids = ids
        self._prefix_cache = cache
        print(f"[ConditionParser] prefix cached: {ids.shape[1]} tokens")

    def _render(self, request: str) -> str:
        return self.tokenizer.apply_chat_template(
            [
                {"role": "system", "content": self.rules},
                {"role": "user", "content": request},
            ],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )

    # ---------- generation (worker thread only) ----------

    def _generate(self, request: str) -> dict:
        inputs = self.tokenizer(self._render(request), return_tensors="pt").to(self.model.device)
        ids = inputs.input_ids

        extra = {}
        if self._prefix_cache is not None:
            p = self._prefix_ids.shape[1]
            # Tokens must match exactly, otherwise the cache is invalid
            if ids.shape[1] > p and torch.equal(ids[0, :p], self._prefix_ids[0]):
                extra["past_key_values"] = copy.deepcopy(self._prefix_cache)

        if USE_ENFORCER:
            extra["prefix_allowed_tokens_fn"] = build_transformers_prefix_allowed_tokens_fn(
                self._tok_data, JsonSchemaParser(SCHEMA)
            )

        with torch.no_grad():
            output = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
                **extra,
            )

        raw = self.tokenizer.decode(output[0][ids.shape[1]:], skip_special_tokens=True).strip()

        try:
            result = json.loads(_extract_json(raw))
            validate(result, SCHEMA)
            return result
        except json.JSONDecodeError:
            return {"error": "Model returned invalid JSON", "raw": raw}
        except ValidationError as e:
            return {"error": f"Schema violation: {e.message}", "raw": raw}

    def _loop(self):
        while True:
            item = self._q.get()
            if item is _STOP:
                break
            request, fut = item
            if not fut.set_running_or_notify_cancel():
                continue
            try:
                fut.set_result(self._generate(request))
            except Exception as e:
                fut.set_exception(e)

    # ---------- public API (thread-safe) ----------

    def submit(self, request: str) -> Future:
        fut: Future = Future()
        self._q.put((request, fut))
        return fut

    def understand_condition(self, request: str, timeout: float | None = None) -> dict:
        return self.submit(request).result(timeout=timeout)

    async def understand_condition_async(self, request: str) -> dict:
        return await asyncio.wrap_future(self.submit(request))

    def pending(self) -> int:
        return self._q.qsize()

    def close(self):
        self._q.put(_STOP)
        self._worker.join()


parser = ConditionParser(model, tokenizer, RULES)
understand_condition = parser.understand_condition

[ConditionParser] prefix cached: 3836 tokens


In [ ]:
import re

# Works in both Python `re` and .NET Regex
ANY = r"[\s\S]*"     # '.' does not cross newlines — WhatsApp messages can be multi-line
DIGIT = "[0-9]"      # \d also matches Arabic-Indic and other Unicode digits
LETTER = "[A-Za-zא-תء-يА-яЁё]"  # Latin, Hebrew, Arabic, Russian

SEPARATORS = {
    "hyphen": "-",
    "colon": ":",
    "slash": "/",
}


def escape_regex(value):
    if value is None:
        return ""
    return re.sub(r'([.*+?^${}()|[\]\\])', r'\\\1', value)


def _quant(count, kind):
    if count is None:
        return "+"                                  # no count = one or more
    if not isinstance(count, int) or count < 1:
        raise ValueError(f"Invalid {kind} count: {count}")
    return f"{{{count}}}"


def _component(condition):
    """A piece that sits immediately in place inside a sequence."""
    t = condition.get("type")
    value = condition.get("value")
    count = condition.get("count")

    if t in ("starts_with", "literal", "exact", "contains", "contains_after", "ends_with"):
        return escape_regex(value)
    if t == "space":
        return " "
    if t == "digits":
        return DIGIT + _quant(count, "digits")
    if t == "letters":
        return LETTER + _quant(count, "letters")
    if t == "one_of":
        if not isinstance(value, list) or not value:
            raise ValueError("one_of requires values")
        return "(?:" + "|".join(escape_regex(v) for v in value) + ")"
    if t == "separator":
        symbol = SEPARATORS.get(value)
        if symbol is None:
            raise ValueError(f"Unsupported separator: {value}")
        return escape_regex(symbol)
    raise ValueError(f"Unsupported condition type: {t}")


def compile_condition(data):
    conditions = data.get("conditions", [])
    whole_input = data.get("whole_input", False)

    # not_contains -> lookaheads, always anchored at ^ (unanchored they block nothing)
    looks = [
        "(?!" + ANY + escape_regex(c.get("value")) + ")"
        for c in conditions if c.get("type") == "not_contains"
    ]
    rest = [c for c in conditions if c.get("type") != "not_contains"]

    # ---- closed format ----
    if whole_input:
        body = ""
        for c in rest:
            t = c.get("type")
            if t in ("contains", "contains_after"):
                body += ANY + _component(c) + ANY
            elif t == "ends_with":
                body += ANY + _component(c)
            else:
                body += _component(c)
        return "^" + "".join(looks) + body + "$"

    # ---- partial: ordered chain + lookaheads ----
    chain = ""
    anchored = False
    tail = ""

    for i, c in enumerate(rest):
        t = c.get("type")
        nxt = rest[i + 1].get("type") if i + 1 < len(rest) else None

        if t == "starts_with" and i == 0:
            chain, anchored = escape_regex(c.get("value")), True
        elif t == "contains":
            if nxt == "contains_after":            # start of an ordered chain
                chain += (ANY if chain else "") + escape_regex(c.get("value"))
            else:                                   # unordered — any position
                looks.append("(?=" + ANY + escape_regex(c.get("value")) + ")")
        elif t == "contains_after":
            chain += ANY + escape_regex(c.get("value"))
        elif t == "ends_with":
            tail = escape_regex(c.get("value")) + "$"
        elif t == "exact":
            return "^" + "".join(looks) + escape_regex(c.get("value")) + "$"
        else:                                       # immediate structured component
            chain += _component(c)

    if tail and (chain or anchored):
        tail = ANY + tail

    if anchored:
        return "^" + "".join(looks) + chain + tail
    if looks:
        return "^" + "".join(looks) + (ANY if (chain or tail) else "") + chain + tail
    return chain + tail


# ----------------------------------------------------------------------

# Longest names first so "شرطة مائلة" wins over "شرطة", "קו תחתון" etc.
SYMBOL_WORDS = sorted({
    # Hebrew
    "נקודתיים": ":", "מקף": "-", "לוכסן": "/", "נקודה": ".", "קו תחתון": "_", "שטרודל": "@",
    # English
    "colon": ":", "hyphen": "-", "dash": "-", "slash": "/", "dot": ".", "period": ".",
    "underscore": "_", "at sign": "@",
    # Russian
    "двоеточие": ":", "дефис": "-", "тире": "-", "слэш": "/", "точка": ".",
    "подчёркивание": "_", "собака": "@",
    # Arabic
    "نقطتان": ":", "شرطة مائلة": "/", "شرطة سفلية": "_", "شرطة": "-", "نقطة": ".",
}.items(), key=lambda kv: -len(kv[0]))

SYMBOL_CHARS = {".", ":", "-", "/", "_", "@", "}", "{", "–", "—"}


def _requested_symbols(request):
    """Symbols the user named, in the order they appear in the request (quoted text ignored)."""
    text = re.sub(r'"[^"]*"', lambda m: " " * len(m.group()), request.lower())
    found = []
    taken = [False] * len(text)
    for word, symbol in SYMBOL_WORDS:
        for m in re.finditer(re.escape(word.lower()), text):
            if not any(taken[m.start():m.end()]):
                found.append((m.start(), symbol))
                for k in range(m.start(), m.end()):
                    taken[k] = True
    return [s for _, s in sorted(found)]


def normalize_dsl(data, request):
    if not isinstance(data, dict):
        return data

    conditions = data.get("conditions")
    if not isinstance(conditions, list):
        return data

    # 1. one_of with a single option: exact if alone, literal inside a sequence
    for condition in conditions:
        if (
            condition.get("type") == "one_of"
            and isinstance(condition.get("value"), list)
            and len(condition["value"]) == 1
        ):
            condition["value"] = condition["value"][0]
            if len(conditions) == 1:
                condition["type"] = "exact"
                data["whole_input"] = True
            else:
                condition["type"] = "literal"

    # 2. separator -> literal
    for condition in conditions:
        if condition.get("type") == "separator":
            condition["type"] = "literal"
            condition["value"] = SEPARATORS.get(condition.get("value"), condition.get("value"))

    # 3. fix symbols the model swapped — match by order of appearance
    requested = _requested_symbols(request)
    symbol_slots = [
        c for c in conditions
        if c.get("type") == "literal"
        and isinstance(c.get("value"), str)
        and c["value"] in SYMBOL_CHARS
    ]
    if requested and len(requested) == len(symbol_slots):
        for slot, symbol in zip(symbol_slots, requested):
            slot["value"] = symbol

    return data

In [ ]:
import json

def normalize(data):
    return json.dumps(data, ensure_ascii=False, sort_keys=True)

## בדיקה מהירה

In [ ]:
req = 'התגובה מתחילה ב"ID", אחר כך נקודתיים ואז 4 ספרות'
dsl = normalize_dsl(understand_condition(req), req)
print(json.dumps(dsl, ensure_ascii=False, indent=2))
print("regex:", compile_condition(dsl))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{
  "conditions": [
    {
      "type": "starts_with",
      "value": "ID",
      "count": null
    },
    {
      "type": "literal",
      "value": "։",
      "count": null
    },
    {
      "type": "digits",
      "value": null,
      "count": 4
    }
  ],
  "whole_input": true
}
regex: ^ID։[0-9]{4}$


## טסטים

In [ ]:
TEST_CASES = [

    # ============================================================
    # HEBREW — 50 hard-coded tests
    # ============================================================

    {
        "id": 1,
        "lang": "he",
        "request": 'הטקסט חייב להתחיל ב"שלום"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "שלום", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 2,
        "lang": "he",
        "request": 'ההודעה צריכה להתחיל במילה "הזמנה"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "הזמנה", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 3,
        "lang": "he",
        "request": 'הטקסט צריך להסתיים ב"תודה"',
        "expected": {
            "conditions": [
                {"type": "ends_with", "value": "תודה", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 4,
        "lang": "he",
        "request": 'בסוף ההודעה חייב להופיע "DONE"',
        "expected": {
            "conditions": [
                {"type": "ends_with", "value": "DONE", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 5,
        "lang": "he",
        "request": 'הטקסט חייב להכיל "אישור"',
        "expected": {
            "conditions": [
                {"type": "contains", "value": "אישור", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 6,
        "lang": "he",
        "request": 'איפשהו בהודעה צריכה להופיע המילה "הצלחה"',
        "expected": {
            "conditions": [
                {"type": "contains", "value": "הצלחה", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 7,
        "lang": "he",
        "request": 'התגובה חייבת להיות בדיוק "כן"',
        "expected": {
            "conditions": [
                {"type": "exact", "value": "כן", "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 8,
        "lang": "he",
        "request": 'רק המילה "מאושר" מותרת בתגובה',
        "expected": {
            "conditions": [
                {"type": "exact", "value": "מאושר", "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 9,
        "lang": "he",
        "request": 'התגובה יכולה להיות רק "כן" או "לא"',
        "expected": {
            "conditions": [
                {"type": "one_of", "value": ["כן", "לא"], "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 10,
        "lang": "he",
        "request": 'קבל רק "אישור", "ביטול" או "המתנה"',
        "expected": {
            "conditions": [
                {
                    "type": "one_of",
                    "value": ["אישור", "ביטול", "המתנה"],
                    "count": None
                }
            ],
            "whole_input": True
        }
    },

    {
        "id": 11,
        "lang": "he",
        "request": 'התגובה חייבת להיות בדיוק 4 ספרות',
        "expected": {
            "conditions": [
                {"type": "digits", "value": None, "count": 4}
            ],
            "whole_input": True
        }
    },

    {
        "id": 12,
        "lang": "he",
        "request": 'אני רוצה לקבל רק מספר שמורכב בדיוק מ-8 ספרות',
        "expected": {
            "conditions": [
                {"type": "digits", "value": None, "count": 8}
            ],
            "whole_input": True
        }
    },

    {
        "id": 13,
        "lang": "he",
        "request": 'התגובה חייבת להכיל בדיוק 3 אותיות בלבד',
        "expected": {
            "conditions": [
                {"type": "letters", "value": None, "count": 3}
            ],
            "whole_input": True
        }
    },

    {
        "id": 14,
        "lang": "he",
        "request": 'הטקסט מתחיל ב"שלום" ובהמשך מופיע "עולם"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "שלום", "count": None},
                {"type": "contains_after", "value": "עולם", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 15,
        "lang": "he",
        "request": 'קודם ההודעה מתחילה ב"START" ובהמשך צריכה להופיע "END"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "START", "count": None},
                {"type": "contains_after", "value": "END", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 16,
        "lang": "he",
        "request": 'התגובה מתחילה ב"קוד", אחר כך רווח ואז בדיוק 5 ספרות',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "קוד", "count": None},
                {"type": "space", "value": None, "count": None},
                {"type": "digits", "value": None, "count": 5}
            ],
            "whole_input": True
        }
    },

    {
        "id": 17,
        "lang": "he",
        "request": 'הטקסט חייב להתחיל ב"ORDER", אחריו מקף ואז בדיוק 6 ספרות',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "ORDER", "count": None},
                {"type": "literal", "value": "-", "count": None},
                {"type": "digits", "value": None, "count": 6}
            ],
            "whole_input": True
        }
    },

    {
        "id": 18,
        "lang": "he",
        "request": 'התגובה מתחילה ב"ID", אחר כך נקודתיים ואז 4 ספרות',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "ID", "count": None},
                {"type": "literal", "value": ":", "count": None},
                {"type": "digits", "value": None, "count": 4}
            ],
            "whole_input": True
        }
    },

    {
        "id": 19,
        "lang": "he",
        "request": 'הטקסט מתחיל ב"ABC", אחר כך לוכסן ואז 3 ספרות',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "ABC", "count": None},
                {"type": "literal", "value": "/", "count": None},
                {"type": "digits", "value": None, "count": 3}
            ],
            "whole_input": True
        }
    },

    {
        "id": 20,
        "lang": "he",
        "request": 'הטקסט חייב להתחיל ב"שלום" אבל אסור שיופיע בו "להתראות"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "שלום", "count": None},
                {"type": "not_contains", "value": "להתראות", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 21,
        "lang": "he",
        "request": 'ההודעה יכולה להכיל כל דבר אבל אסור שתכיל "ERROR"',
        "expected": {
            "conditions": [
                {"type": "not_contains", "value": "ERROR", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 22,
        "lang": "he",
        "request": 'התגובה חייבת להתחיל ב"טופס" ואחריו רווח ו-7 ספרות בדיוק',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "טופס", "count": None},
                {"type": "space", "value": None, "count": None},
                {"type": "digits", "value": None, "count": 7}
            ],
            "whole_input": True
        }
    },

    {
        "id": 23,
        "lang": "he",
        "request": 'אני מצפה לתגובה שהיא רק "OK" או "FAIL"',
        "expected": {
            "conditions": [
                {"type": "one_of", "value": ["OK", "FAIL"], "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 24,
        "lang": "he",
        "request": 'בתחילת הטקסט חייב להיות "USER"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "USER", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 25,
        "lang": "he",
        "request": 'בסוף התגובה אני רוצה לראות "SUCCESS"',
        "expected": {
            "conditions": [
                {"type": "ends_with", "value": "SUCCESS", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 26,
        "lang": "he",
        "request": 'התשובה היחידה שמותר לקבל היא "בוצע"',
        "expected": {
            "conditions": [
                {"type": "exact", "value": "בוצע", "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 27,
        "lang": "he",
        "request": 'התשובה צריכה להיות אחת מהאפשרויות "א", "ב" או "ג"',
        "expected": {
            "conditions": [
                {"type": "one_of", "value": ["א", "ב", "ג"], "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 28,
        "lang": "he",
        "request": 'אני צריך שההודעה תכיל את הטקסט "123-ABC"',
        "expected": {
            "conditions": [
                {"type": "contains", "value": "123-ABC", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 29,
        "lang": "he",
        "request": 'ההודעה מתחילה ב"לקוח", אחריה רווח ואז "פעיל"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "לקוח", "count": None},
                {"type": "space", "value": None, "count": None},
                {"type": "literal", "value": "פעיל", "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 30,
        "lang": "he",
        "request": 'הטקסט צריך להתחיל ב"A" ובהמשך להכיל "B"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "A", "count": None},
                {"type": "contains_after", "value": "B", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 31,
        "lang": "he",
        "request": 'רק 6 ספרות מותרות בתגובה',
        "expected": {
            "conditions": [
                {"type": "digits", "value": None, "count": 6}
            ],
            "whole_input": True
        }
    },

    {
        "id": 32,
        "lang": "he",
        "request": 'התגובה היא בדיוק אחת מהאפשרויות "1", "2", "3" או "4"',
        "expected": {
            "conditions": [
                {"type": "one_of", "value": ["1", "2", "3", "4"], "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 33,
        "lang": "he",
        "request": 'הטקסט חייב להכיל "שלום" ואסור שיכיל "שגיאה"',
        "expected": {
            "conditions": [
                {"type": "contains", "value": "שלום", "count": None},
                {"type": "not_contains", "value": "שגיאה", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 34,
        "lang": "he",
        "request": 'ההודעה חייבת להתחיל ב"REF", אחר כך מקף ואז 8 ספרות',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "REF", "count": None},
                {"type": "literal", "value": "-", "count": None},
                {"type": "digits", "value": None, "count": 8}
            ],
            "whole_input": True
        }
    },

    {
        "id": 35,
        "lang": "he",
        "request": 'התגובה חייבת להסתיים ב"מאושר"',
        "expected": {
            "conditions": [
                {"type": "ends_with", "value": "מאושר", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 36,
        "lang": "he",
        "request": 'חייב להופיע "ABC" איפשהו בתגובה',
        "expected": {
            "conditions": [
                {"type": "contains", "value": "ABC", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 37,
        "lang": "he",
        "request": 'אסור שהתגובה תכיל את המילה "ביטול"',
        "expected": {
            "conditions": [
                {"type": "not_contains", "value": "ביטול", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 38,
        "lang": "he",
        "request": 'התגובה צריכה להיות בדיוק "פעיל" או "לא פעיל"',
        "expected": {
            "conditions": [
                {"type": "one_of", "value": ["פעיל", "לא פעיל"], "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 39,
        "lang": "he",
        "request": 'הטקסט מתחיל ב"X", אחריו נקודתיים ואז בדיוק 2 ספרות',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "X", "count": None},
                {"type": "literal", "value": ":", "count": None},
                {"type": "digits", "value": None, "count": 2}
            ],
            "whole_input": True
        }
    },

    {
        "id": 40,
        "lang": "he",
        "request": 'הטקסט מתחיל ב"HELLO", ובהמשך חייב להיות "WORLD", אבל אסור שיהיה "ERROR"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "HELLO", "count": None},
                {"type": "contains_after", "value": "WORLD", "count": None},
                {"type": "not_contains", "value": "ERROR", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 41,
        "lang": "he",
        "request": 'אני רוצה בדיוק 9 ספרות בלי שום דבר נוסף',
        "expected": {
            "conditions": [
                {"type": "digits", "value": None, "count": 9}
            ],
            "whole_input": True
        }
    },

    {
        "id": 42,
        "lang": "he",
        "request": 'התגובה חייבת להתחיל ב"מספר", אחריו רווח ואז 3 ספרות בדיוק',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "מספר", "count": None},
                {"type": "space", "value": None, "count": None},
                {"type": "digits", "value": None, "count": 3}
            ],
            "whole_input": True
        }
    },

    {
        "id": 43,
        "lang": "he",
        "request": 'מותר לענות רק "ראשון", "שני" או "שלישי"',
        "expected": {
            "conditions": [
                {
                    "type": "one_of",
                    "value": ["ראשון", "שני", "שלישי"],
                    "count": None
                }
            ],
            "whole_input": True
        }
    },

    {
        "id": 44,
        "lang": "he",
        "request": 'הטקסט צריך להסתיים ב"123"',
        "expected": {
            "conditions": [
                {"type": "ends_with", "value": "123", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 45,
        "lang": "he",
        "request": 'ההודעה חייבת להתחיל ב"#"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "#", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 46,
        "lang": "he",
        "request": 'התגובה חייבת להיות בדיוק "A/B"',
        "expected": {
            "conditions": [
                {"type": "exact", "value": "A/B", "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 47,
        "lang": "he",
        "request": 'הטקסט מכיל "START" ובהמשך צריך להופיע "STOP"',
        "expected": {
            "conditions": [
                {"type": "contains", "value": "START", "count": None},
                {"type": "contains_after", "value": "STOP", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 48,
        "lang": "he",
        "request": 'התגובה מתחילה ב"קוד", רווח, 4 ספרות, מקף ואז 2 ספרות',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "קוד", "count": None},
                {"type": "space", "value": None, "count": None},
                {"type": "digits", "value": None, "count": 4},
                {"type": "literal", "value": "-", "count": None},
                {"type": "digits", "value": None, "count": 2}
            ],
            "whole_input": True
        }
    },

    {
        "id": 49,
        "lang": "he",
        "request": 'התשובה חייבת להיות "פתוח" או "סגור" בלבד',
        "expected": {
            "conditions": [
                {"type": "one_of", "value": ["פתוח", "סגור"], "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 50,
        "lang": "he",
        "request": 'הטקסט מתחיל ב"BEGIN", בהמשך מופיע "DATA" ובסוף חייב להיות "END"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "BEGIN", "count": None},
                {"type": "contains_after", "value": "DATA", "count": None},
                {"type": "ends_with", "value": "END", "count": None}
            ],
            "whole_input": False
        }
    },
]

In [ ]:
TEST_CASES_EXTRA = [

    # ============================================================
    # EDGE CASES — what the original rules did not cover
    # ============================================================

    # --- A. digits / letters without count ---
    {
        "id": 51, "lang": "he",
        "request": 'רק מספרים',
        "expected": {"conditions": [
            {"type": "digits", "value": None, "count": None}
        ], "whole_input": True}
    },
    {
        "id": 52, "lang": "he",
        "request": 'התגובה חייבת להכיל רק אותיות',
        "expected": {"conditions": [
            {"type": "letters", "value": None, "count": None}
        ], "whole_input": True}
    },

    # --- C. one_of inside sequence ---
    {
        "id": 53, "lang": "he",
        "request": 'מתחיל ב"A" או "B" ואז 3 ספרות',
        "expected": {"conditions": [
            {"type": "one_of", "value": ["A", "B"], "count": None},
            {"type": "digits", "value": None, "count": 3}
        ], "whole_input": True}
    },
    {
        "id": 54, "lang": "he",
        "request": 'ההודעה מתחילה ב"ID", אחריה רווח ואז "פעיל" או "מושהה"',
        "expected": {"conditions": [
            {"type": "starts_with", "value": "ID", "count": None},
            {"type": "space", "value": None, "count": None},
            {"type": "one_of", "value": ["פעיל", "מושהה"], "count": None}
        ], "whole_input": True}
    },

    # --- D. sequence starting with digits/letters, literal at end ---
    {
        "id": 55, "lang": "he",
        "request": 'התגובה היא 3 ספרות, אחריהן מקף ואז "IL"',
        "expected": {"conditions": [
            {"type": "digits", "value": None, "count": 3},
            {"type": "literal", "value": "-", "count": None},
            {"type": "literal", "value": "IL", "count": None}
        ], "whole_input": True}
    },
    {
        "id": 56, "lang": "he",
        "request": 'התגובה צריכה להיות 5 אותיות ואז 2 ספרות',
        "expected": {"conditions": [
            {"type": "letters", "value": None, "count": 5},
            {"type": "digits", "value": None, "count": 2}
        ], "whole_input": True}
    },

    # --- E. quoted value containing "או" ---
    {
        "id": 57, "lang": "he",
        "request": 'התגובה חייבת להיות בדיוק "כן או לא"',
        "expected": {"conditions": [
            {"type": "exact", "value": "כן או לא", "count": None}
        ], "whole_input": True}
    },

    # --- F. immediate next component, no space ---
    {
        "id": 58, "lang": "he",
        "request": 'הטקסט מתחיל ב"קוד", ואחריו 4 ספרות',
        "expected": {"conditions": [
            {"type": "starts_with", "value": "קוד", "count": None},
            {"type": "digits", "value": None, "count": 4}
        ], "whole_input": True}
    },

    # --- G. extra symbols ---
    {
        "id": 59, "lang": "he",
        "request": 'ההודעה מתחילה ב"FILE", אחריה קו תחתון ואז 4 ספרות',
        "expected": {"conditions": [
            {"type": "starts_with", "value": "FILE", "count": None},
            {"type": "literal", "value": "_", "count": None},
            {"type": "digits", "value": None, "count": 4}
        ], "whole_input": True}
    },
    {
        "id": 60, "lang": "he",
        "request": 'התגובה היא 2 ספרות, נקודה ואז 2 ספרות',
        "expected": {"conditions": [
            {"type": "digits", "value": None, "count": 2},
            {"type": "literal", "value": ".", "count": None},
            {"type": "digits", "value": None, "count": 2}
        ], "whole_input": True}
    },
    {
        "id": 61, "lang": "he",
        "request": 'הטקסט מתחיל ב"user", אחריו שטרודל ואז "mail.com"',
        "expected": {"conditions": [
            {"type": "starts_with", "value": "user", "count": None},
            {"type": "literal", "value": "@", "count": None},
            {"type": "literal", "value": "mail.com", "count": None}
        ], "whole_input": True}
    },

    # --- multiple negatives / order with ends_with ---
    {
        "id": 62, "lang": "he",
        "request": 'אסור שהתגובה תכיל "test" ו"debug"',
        "expected": {"conditions": [
            {"type": "not_contains", "value": "test", "count": None},
            {"type": "not_contains", "value": "debug", "count": None}
        ], "whole_input": False}
    },
    {
        "id": 63, "lang": "he",
        "request": 'הטקסט חייב להסתיים ב"תודה" ואסור שיכיל "שגיאה"',
        "expected": {"conditions": [
            {"type": "ends_with", "value": "תודה", "count": None},
            {"type": "not_contains", "value": "שגיאה", "count": None}
        ], "whole_input": False}
    },

    # ============================================================
    # OTHER LANGUAGES — not covered by the Hebrew set
    # ============================================================

    {
        "id": 64, "lang": "en",
        "request": 'The response must be exactly "yes" or "no"',
        "expected": {"conditions": [
            {"type": "one_of", "value": ["yes", "no"], "count": None}
        ], "whole_input": True}
    },
    {
        "id": 65, "lang": "en",
        "request": 'Starts with "INV", then a hyphen, then exactly 5 digits',
        "expected": {"conditions": [
            {"type": "starts_with", "value": "INV", "count": None},
            {"type": "literal", "value": "-", "count": None},
            {"type": "digits", "value": None, "count": 5}
        ], "whole_input": True}
    },
    {
        "id": 66, "lang": "ru",
        "request": 'Ответ должен содержать только 4 цифры',
        "expected": {"conditions": [
            {"type": "digits", "value": None, "count": 4}
        ], "whole_input": True}
    },
    {
        "id": 67, "lang": "ru",
        "request": 'Текст начинается с "Привет", а затем содержит "мир"',
        "expected": {"conditions": [
            {"type": "starts_with", "value": "Привет", "count": None},
            {"type": "contains_after", "value": "мир", "count": None}
        ], "whole_input": False}
    },
    {
        "id": 68, "lang": "ar",
        "request": 'يجب أن يكون الرد 6 أرقام فقط',
        "expected": {"conditions": [
            {"type": "digits", "value": None, "count": 6}
        ], "whole_input": True}
    },
    {
        "id": 69, "lang": "ar",
        "request": 'يبدأ النص بـ "مرحبا"',
        "expected": {"conditions": [
            {"type": "starts_with", "value": "مرحبا", "count": None}
        ], "whole_input": False}
    },
    {
        "id": 70, "lang": "ar",
        "request": 'يجب ألا يحتوي النص على "خطأ"',
        "expected": {"conditions": [
            {"type": "not_contains", "value": "خطأ", "count": None}
        ], "whole_input": False}
    },
]

In [ ]:
import re, time

def run_tests(cases, verbose=False):
    results = []
    t0 = time.time()
    for test in cases:
        try:
            actual = normalize_dsl(understand_condition(test["request"]), test["request"])
            passed = normalize(actual) == normalize(test["expected"])
            pattern = compile_condition(actual) if "error" not in actual else None
            if pattern:
                re.compile(pattern)
        except Exception as e:
            actual, passed, pattern = {"error": str(e)}, False, None

        results.append({**test, "actual": actual, "passed": passed, "regex": pattern})
        mark = "✅" if passed else "❌"
        print(f'{mark} #{test["id"]:>3} [{test["lang"]}] {test["request"]}')
        if verbose or not passed:
            print("   regex:   ", pattern)
        if not passed:
            print("   expected:", json.dumps(test["expected"], ensure_ascii=False))
            print("   actual:  ", json.dumps(actual, ensure_ascii=False))

    elapsed = time.time() - t0
    ok = sum(r["passed"] for r in results)
    print(f"\nPassed: {ok}/{len(results)} ({ok / len(results) * 100:.1f}%)"
          f" | {elapsed:.1f}s total, {elapsed / len(results):.2f}s/request")
    return results


results = run_tests(TEST_CASES)            # 1-50

✅ #  1 [he] הטקסט חייב להתחיל ב"שלום"
✅ #  2 [he] ההודעה צריכה להתחיל במילה "הזמנה"
✅ #  3 [he] הטקסט צריך להסתיים ב"תודה"
✅ #  4 [he] בסוף ההודעה חייב להופיע "DONE"
✅ #  5 [he] הטקסט חייב להכיל "אישור"
✅ #  6 [he] איפשהו בהודעה צריכה להופיע המילה "הצלחה"
✅ #  7 [he] התגובה חייבת להיות בדיוק "כן"
✅ #  8 [he] רק המילה "מאושר" מותרת בתגובה
✅ #  9 [he] התגובה יכולה להיות רק "כן" או "לא"
✅ # 10 [he] קבל רק "אישור", "ביטול" או "המתנה"
✅ # 11 [he] התגובה חייבת להיות בדיוק 4 ספרות
✅ # 12 [he] אני רוצה לקבל רק מספר שמורכב בדיוק מ-8 ספרות
✅ # 13 [he] התגובה חייבת להכיל בדיוק 3 אותיות בלבד
✅ # 14 [he] הטקסט מתחיל ב"שלום" ובהמשך מופיע "עולם"
✅ # 15 [he] קודם ההודעה מתחילה ב"START" ובהמשך צריכה להופיע "END"
✅ # 16 [he] התגובה מתחילה ב"קוד", אחר כך רווח ואז בדיוק 5 ספרות
✅ # 17 [he] הטקסט חייב להתחיל ב"ORDER", אחריו מקף ואז בדיוק 6 ספרות
❌ # 18 [he] התגובה מתחילה ב"ID", אחר כך נקודתיים ואז 4 ספרות
   regex:    ^ID։[0-9]{4}$
   expected: {"conditions": [{"type": "starts_with", "value": "ID", "count"

In [ ]:
results_extra = run_tests(TEST_CASES_EXTRA)   # 51-70

✅ # 51 [he] רק מספרים
✅ # 52 [he] התגובה חייבת להכיל רק אותיות
✅ # 53 [he] מתחיל ב"A" או "B" ואז 3 ספרות
✅ # 54 [he] ההודעה מתחילה ב"ID", אחריה רווח ואז "פעיל" או "מושהה"
✅ # 55 [he] התגובה היא 3 ספרות, אחריהן מקף ואז "IL"
✅ # 56 [he] התגובה צריכה להיות 5 אותיות ואז 2 ספרות
✅ # 57 [he] התגובה חייבת להיות בדיוק "כן או לא"
✅ # 58 [he] הטקסט מתחיל ב"קוד", ואחריו 4 ספרות
✅ # 59 [he] ההודעה מתחילה ב"FILE", אחריה קו תחתון ואז 4 ספרות
✅ # 60 [he] התגובה היא 2 ספרות, נקודה ואז 2 ספרות
✅ # 61 [he] הטקסט מתחיל ב"user", אחריו שטרודל ואז "mail.com"
✅ # 62 [he] אסור שהתגובה תכיל "test" ו"debug"
✅ # 63 [he] הטקסט חייב להסתיים ב"תודה" ואסור שיכיל "שגיאה"
✅ # 64 [en] The response must be exactly "yes" or "no"
✅ # 65 [en] Starts with "INV", then a hyphen, then exactly 5 digits
✅ # 66 [ru] Ответ должен содержать только 4 цифры
❌ # 67 [ru] Текст начинается с "Привет", а затем содержит "мир"
   regex:    ^(?=[\s\S]*мир)Привет
   expected: {"conditions": [{"type": "starts_with", "value": "Привет", "count":

## בדיקת regex מול הודעות

In [ ]:
def check(request, samples):
    dsl = normalize_dsl(understand_condition(request), request)
    pattern = compile_condition(dsl)
    print("regex:", pattern)
    for s in samples:
        print("  ✅" if re.search(pattern, s) else "  ❌", repr(s))

check('ההודעה יכולה להכיל כל דבר אבל אסור שתכיל "ERROR"', ["הכל תקין", "יש ERROR כאן"])
check('התגובה חייבת להכיל בדיוק 3 אותיות בלבד', ["אבג", "abc", "ab1"])

regex: ^(?![\s\S]*ERROR)
  ✅ 'הכל תקין'
  ❌ 'יש ERROR כאן'
regex: ^[A-Za-zא-תء-يА-яЁё]{3}$
  ✅ 'אבג'
  ✅ 'abc'
  ❌ 'ab1'


In [ ]:
import gc
gc.collect(); torch.cuda.empty_cache()
print("allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB | reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

allocated: 6.2 GB | reserved: 7.16 GB
